In [13]:
!pip install pytest-playwright

In [14]:
!playwright install

In [15]:
import nest_asyncio
import asyncio
import csv
import re
from playwright.async_api import async_playwright

nest_asyncio.apply()

async def scrape_lelong_final_fix(target_goal=200000):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(user_agent="Mozilla/5.0...")
        page = await context.new_page()

        all_data = []
        page_num = 1
        base_url = "https://www.lelong.my/catalog/all/list?TheKeywords=autocount"

        while len(all_data) < target_goal:
            print(f"Scraping Page {page_num}... Found: {len(all_data)}")
            await page.goto(f"{base_url}&P={page_num}", wait_until="networkidle")
            
            # Wait for the item cards
            await page.wait_for_selector(".item", timeout=10000)
            items = await page.query_selector_all(".item")
            
            if not items: break

            for item in items:
                try:
                    # 1. Product Name
                    name_el = await item.query_selector(".title")
                    p_name = (await name_el.inner_text()).strip().split('\n')[0]

                    # 2. STORE NAME (Targeting the house icon container)
                    # We look for the link that is a child of the store-info or username area
                    # This targets the specific 'a' tag next to the icon
                    # store_el = await item.query_selector(".store-name a, .username a, a[href*='/merchant/'], .seller-info a")
                    # s_name = (await store_el.inner_text()).strip() if store_el else "Unknown"
                    
                    # If it's still unknown, we grab the text after 'By' or 'Store:'
                    # if s_name == "Unknown":
                    #     raw_text = await item.inner_text()
                    #     match = re.search(r"(?:By|Store)\s*:\s*([^\n,]+)", raw_text, re.I)
                    #     if match: s_name = match.group(1).strip()

                    # 3. PRICE (Targeting the BOLD price only)
                    # We specifically exclude <s> tags (original price)
                    price_el = await item.query_selector("b.price, span.price b, .price-box strong:not(s *)")
                    price_raw = await price_el.inner_text() if price_el else "0"
                    price_clean = re.sub(r"[^\d.]", "", price_raw)

                    # 4. Ship From & Arrival
                    card_text = await item.inner_text()
                    ship_from = "Malaysia"
                    arrival = "1-3 working days"
                    
                    if "Ship from :" in card_text:
                        ship_from = card_text.split("Ship from :")[1].split(",")[0].strip()
                    if "Estimated Arrival" in card_text:
                        arrival = card_text.split("Estimated Arrival")[1].split("\n")[0].strip()

                    # 5. Shipping Fee (Beside the lorry)
                    ship_fee = "Free"
                    if "FREE" in card_text.upper():
                        ship_fee = "Free"
                    else:
                        # Find all RM amounts and pick the smallest/last one (usually shipping)
                        rm_values = re.findall(r"RM\s*(\d+\.\d{2})", card_text)
                        if len(rm_values) > 1:
                            ship_fee = rm_values[-1]

                    all_data.append({
                        "product_name": p_name,
                        # "store_name": s_name,
                        "ship_from": ship_from,
                        "estimated_arrival": arrival,
                        "price": price_clean,
                        "shipping_fee": ship_fee
                    })

                    if len(all_data) >= target_goal: break
                except:
                    continue
            
            page_num += 1

        save_to_csv(all_data)
        await browser.close()

def save_to_csv(data):
    if not data: return
    keys = data[0].keys()
    with open('lelong_raw_data.csv', 'w', newline='', encoding='utf-8') as f:
        dict_writer = csv.DictWriter(f, fieldnames=keys)
        dict_writer.writeheader()
        writerows = data
        dict_writer.writerows(writerows)
    print(f"Success! Saved {len(data)} records.")

await scrape_lelong_final_fix(target_goal=200000)

In [24]:
# import asyncio
# import csv
# import re
# import nest_asyncio
# from playwright.async_api import async_playwright

# # Essential for Jupyter Notebooks
# nest_asyncio.apply()

# async def run_baseline_crawler(target_records=100):
#     async with async_playwright() as p:
#         # headless=False allows you to see the browser and verify the store name is visible
#         browser = await p.chromium.launch(headless=False)
#         context = await browser.new_context(viewport={'width': 1280, 'height': 800})
#         page = await context.new_page()
        
#         all_data = []
#         page_num = 1
#         base_url = "https://www.lelong.my/catalog/all/list?TheKeywords=electronics"

#         print(f"🚀 Starting crawl for {target_records} records...")

#         while len(all_data) < target_records:
#             try:
#                 print(f"📂 Accessing Page {page_num}...")
#                 # Use 'networkidle' to ensure store names (often loaded via JS) appear
#                 await page.goto(f"{base_url}&P={page_num}", wait_until="networkidle", timeout=60000)
                
#                 # Wait for the item container
#                 await page.wait_for_selector(".item", timeout=15000)
#                 items = await page.query_selector_all(".item")
                
#                 if not items:
#                     print("❌ No items found on this page.")
#                     break

#                 for item in items:
#                     if len(all_data) >= target_records:
#                         break

#                     # 1. Product Name
#                     title_el = await item.query_selector(".title")
#                     product_name = (await title_el.inner_text()).strip() if title_el else "N/A"

#                     # 2. Store Name (Robust Multi-Step Logic)
#                     store_name = "Unknown"
                    
#                     # Method A: Find the link next to the house icon using XPath
#                     # This looks for the anchor tag that is a sibling to the house icon
#                     store_xpath = await item.query_selector("xpath=.//i[contains(@class, 'home') or contains(@class, 'store')]/following-sibling::a")
                    
#                     if store_xpath:
#                         store_name = (await store_xpath.inner_text()).strip()
                    
#                     # Method B Fallback: Look for any link containing 'merchant' in the URL
#                     if store_name == "Unknown" or store_name == "":
#                         merchant_link = await item.query_selector("a[href*='/merchant/']")
#                         if merchant_link:
#                             store_name = (await merchant_link.inner_text()).strip()

#                     # 3. Price
#                     price_el = await item.query_selector(".price b")
#                     price_text = await price_el.inner_text() if price_el else "0.00"
#                     price = re.sub(r"[^\d.]", "", price_text)

#                     # 4. Shipping Fee
#                     ship_el = await item.query_selector(".shipping-info p")
#                     ship_text = await ship_el.inner_text() if ship_el else "0.00"
#                     ship_fee = re.sub(r"[^\d.]", "", ship_text) if "RM" in ship_text else "0.00"

#                     all_data.append({
#                         "product_name": product_name,
#                         "store_name": store_name,
#                         "price": price,
#                         "shipping_fee": ship_fee
#                     })

#                 print(f"✅ Collected {len(all_data)} records total.")
#                 page_num += 1
                
#             except Exception as e:
#                 print(f"⚠️ Error on page {page_num}: {e}")
#                 break

#         await browser.close()
#         return all_data

# async def main():
#     # ADJUST TARGET HERE
#     target = 50 
#     results = await run_baseline_crawler(target_records=target)
    
#     filename = "lelong_final_report.csv"
    
#     if results:
#         headers = results[0].keys()
#         with open(filename, mode="w", newline="", encoding="utf-8") as f:
#             writer = csv.DictWriter(f, fieldnames=headers)
#             writer.writeheader()
#             writer.writerows(results)
            
#         print("\n" + "="*40)
#         print(f"DONE! File saved as: {filename}")
#         print("="*40)
#     else:
#         print("❌ No data was collected. Check your internet or selectors.")

# # Run the script in Jupyter
# await main()

🚀 Starting crawl for 50 records...
📂 Accessing Page 1...
✅ Collected 50 records total.

DONE! File saved as: lelong_final_report.csv
